In [ ]:
import math as _m
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import numpy as np
from PIL import Image
from torchvision import transforms
from accelerate import Accelerator
import timm
import os, math, random, logging
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import json
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.utils as vutils
import matplotlib.pyplot as plt
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL

In [ ]:
#Same modules as Train_Decoder.ipynb
class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1
    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)
    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)
    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock2d(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=None):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.act1  = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.act2  = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip  = nn.Conv2d(in_channels, out_channels, 1) if in_channels!=out_channels else nn.Identity()
        if time_dim is not None:
            self.to_time = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_channels))
        else:
            self.to_time = None

    def forward(self, x, t_emb=None):
        h = self.conv1(self.act1(self.norm1(x)))
        if self.to_time is not None and t_emb is not None:
            h = h + self.to_time(t_emb)[:, :, None, None]
        h = self.conv2(self.act2(self.norm2(h)))
        return h + self.skip(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64),
            nn.AvgPool2d(2),
            ResidualBlock(64, 128),
            nn.AvgPool2d(2),
            ResidualBlock(128, 256),
            nn.AvgPool2d(2),
            nn.Conv2d(256, out_channels, kernel_size=1)
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),
            Transpose(-1, -2),
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)
        tokens = self.proj(feat)
        tokens = self.pos_embed(tokens)
        tokens = self.norm(tokens)
        return tokens

def _g(n, c):
    return _m.gcd(n, c) or 1

class SinCosPos2D(nn.Module):
    def __init__(self, H=64, W=64):
        super().__init__()
        yy, xx = torch.meshgrid(
            torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing='ij'
        )
        pe = torch.stack([
            torch.sin(2*_m.pi*xx), torch.cos(2*_m.pi*xx),
            torch.sin(2*_m.pi*yy), torch.cos(2*_m.pi*yy)
        ], dim=0)
        self.register_buffer("pe", pe.float(), persistent=False)

    def forward(self, B):
        return self.pe.unsqueeze(0).repeat(B, 1, 1, 1)

class ResInceptionDilated(nn.Module):
    def __init__(self, ch, mid=None):
        super().__init__()
        mid = mid or ch // 2

        self.pre = nn.Sequential(
            nn.GroupNorm(_g(8, ch), ch),
            nn.SiLU()
        )
        self.reduce = nn.Conv2d(ch, mid, 1)

        self.b1 = nn.Conv2d(mid, mid, 3, padding=1, dilation=1, groups=1, bias=False)
        self.b2 = nn.Conv2d(mid, mid, 3, padding=2, dilation=2, groups=1, bias=False)
        self.b3 = nn.Conv2d(mid, mid, 3, padding=4, dilation=4, groups=1, bias=False)

        self.fuse = nn.Sequential(
            nn.GroupNorm(_g(8, mid*3), mid*3),
            nn.SiLU(),
            nn.Conv2d(mid*3, ch, 1)
        )

    def forward(self, x):
        h = self.pre(x)
        h = self.reduce(h)
        h1 = self.b1(h)
        h2 = self.b2(h)
        h3 = self.b3(h)
        h  = torch.cat([h1, h2, h3], dim=1)
        h  = self.fuse(h)
        return x + h

class UpStage(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
        self.rb   = ResInceptionDilated(ch, mid=ch//2)
    def forward(self, x):
        x = self.up(x)
        x = self.conv(x)
        x = self.rb(x)
        return x

class LatentAdapter(nn.Module):

    def __init__(self, cz=4, cond_ch=64, width=128,
                 num_blocks_64=3, include_posenc=True):
        super().__init__()
        self.include_posenc = include_posenc
        in_ch = cz + 4
        self.in_conv = nn.Sequential(
            nn.Conv2d(in_ch, width, 3, padding=1),
            nn.GroupNorm(_g(8, width), width),
            nn.SiLU()
        )
        self.blocks64 = nn.Sequential(*[ResInceptionDilated(width, mid=width//2) for _ in range(num_blocks_64)])
        self.up1 = UpStage(width)
        self.up2 = UpStage(width)
        self.up3 = UpStage(width)

        def head():
            return nn.Sequential(
                nn.Conv2d(width, cond_ch, 1),
                nn.GroupNorm(_g(8, cond_ch), cond_ch),
                nn.SiLU()
            )
        self.out64  = head()
        self.out128 = head()
        self.out256 = head()
        self.out512 = head()

        self.posenc = SinCosPos2D(64, 64) if include_posenc else None

    def forward(self, z64):
        """
        z64:    [B,4,64,64]
        mask64: [B,1,64,64] or None
        """
        B, _, H, W = z64.shape
        feats = [z64]


        if self.include_posenc:
            feats.append(self.posenc(B).to(z64.dtype).to(z64.device))

        x = torch.cat(feats, dim=1)
        x = self.in_conv(x)
        x = self.blocks64(x)

        f64  = self.out64(x)
        x128 = self.up1(x)
        f128 = self.out128(x128)
        x256 = self.up2(x128)
        f256 = self.out256(x256)
        x512 = self.up3(x256)
        f512 = self.out512(x512)

        return {"s64": f64, "s128": f128, "s256": f256, "s512": f512}

class PrecomputedCascadeDataset(Dataset):
    def __init__(self, index_jsonl):
        self.items = []
        with open(index_jsonl, "r") as f:
            for line in f:
                path = json.loads(line)["pt"]
                if os.path.exists(path):
                    self.items.append(path)
                else:
                    print(f"[WARN] Missing file, skipped: {path}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        pack = torch.load(self.items[i], map_location="cpu")
        return (
            pack["masked_img"].float(),
            pack["target_img"].float(),
            torch.tensor(pack["bbox"], dtype=torch.float32),
            pack["z_cond"].to(torch.float16)
        )


def timestep_embedding(timesteps, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(0, half, dtype=torch.float32, device=timesteps.device) / half
    )
    args = timesteps.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=1)
    if dim % 2:
        emb = torch.cat([emb, emb[:, :1]], dim=1)
    return emb


class Down(nn.Module):
    def __init__(self, in_ch, out_ch, tdim, cond_ch=0):
        super().__init__()
        self.block1 = ResidualBlock2d(in_ch + cond_ch, out_ch, time_dim=tdim)
        self.block2 = ResidualBlock2d(out_ch, out_ch, time_dim=tdim)
        self.down = nn.AvgPool2d(2)

    def forward(self, x, t_emb, cond=None):
        if cond is not None:
            x = torch.cat([x, cond], dim=1)
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        x_down = self.down(x)
        return x, x_down


class Up(nn.Module):
    def __init__(self, in_ch, out_ch, tdim, cond_ch=0):
        super().__init__()
        self.block1 = ResidualBlock2d(in_ch + cond_ch, out_ch, time_dim=tdim)
        self.block2 = ResidualBlock2d(out_ch, out_ch, time_dim=tdim)

    def forward(self, x, skip, t_emb, cond=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        if cond is not None:
            x = torch.cat([x, cond], dim=1)
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        return x


from timm.models.vision_transformer import Attention as ViTAttention
import torch.nn as nn
import math as _m


def _g(n, c):
    return _m.gcd(n, c) or 1


class TimmAttn2D(nn.Module):
    def __init__(self, dim, num_heads=4, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.norm = nn.GroupNorm(_g(8, dim), dim)
        self.attn = ViTAttention(
            dim=dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=proj_drop,
        )

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.norm(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.attn(x)
        x = x.transpose(1, 2).reshape(B, C, H, W)
        return x

class UNet512(nn.Module):
    def __init__(self, base_ch=128, cond_ch=64, time_dim=256):
        super().__init__()
        ch1, ch2, ch3 = base_ch, base_ch * 2, base_ch * 4
        self.time_mlp = nn.Sequential(
            nn.Linear(320, time_dim), nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )
        self.in_conv = nn.Conv2d(3 + cond_ch, ch1, 3, padding=1)
        self.down1 = Down(ch1, ch1, tdim=time_dim, cond_ch=0)
        self.down2 = Down(ch1, ch2, tdim=time_dim, cond_ch=cond_ch)
        self.down3 = Down(ch2, ch3, tdim=time_dim, cond_ch=cond_ch)
        self.attn64 = TimmAttn2D(dim=ch3, num_heads=4)
        self.mid1 = ResidualBlock2d(ch3 + cond_ch, ch3, time_dim=time_dim)
        self.mid2 = ResidualBlock2d(ch3, ch3, time_dim=time_dim)
        self.up3 = Up(ch3 + ch3, ch2, tdim=time_dim, cond_ch=cond_ch)
        self.up2 = Up(ch2 + ch2, ch1, tdim=time_dim, cond_ch=cond_ch)
        self.up1 = Up(ch1 + ch1, ch1, tdim=time_dim, cond_ch=0)
        self.out_norm = nn.GroupNorm(8, ch1)
        self.out = nn.Conv2d(ch1, 3, 3, padding=1)

    def forward(self, x, timesteps, cond_feats):
        t_emb = self.time_mlp(timestep_embedding(timesteps, 320))
        x = torch.cat([x, cond_feats["s512"]], dim=1)
        x = self.in_conv(x)
        skip1, x = self.down1(x, t_emb, cond=None)
        skip2, x = self.down2(x, t_emb, cond=cond_feats["s256"])
        skip3, x = self.down3(x, t_emb, cond=cond_feats["s128"])
        x = self.attn64(x)
        x = torch.cat([x, cond_feats["s64"]], dim=1)
        x = self.mid1(x, t_emb)
        x = self.mid2(x, t_emb)
        x = self.up3(x, skip3, t_emb, cond=cond_feats["s128"])
        x = self.up2(x, skip2, t_emb, cond=cond_feats["s256"])
        x = self.up1(x, skip1, t_emb, cond=None)
        x = self.out(self.out_norm(x).clamp(-6, 6))
        x = torch.tanh(x)
        return x


In [ ]:
#Inference Entrance

latent_path = "latent_filename"
save_dir = "save_filename"
vae_path = "runwayml/stable-diffusion-v1-5"
ckpt_path = "decoder_checkpoint_path"
os.makedirs(save_dir, exist_ok=True)

# ==== Set up device and accelerator ====
device = torch.device("cuda")
accelerator = Accelerator(mixed_precision="fp16")

# ==== Load diffusion model components ====
adapter = LatentAdapter(cz=4, cond_ch=96)
unet = UNet512(base_ch=192, cond_ch=96, time_dim=256)
scheduler = DDPMScheduler.from_pretrained(vae_path, subfolder="scheduler")
scheduler.set_timesteps(150, device=device)
scheduler.config.prediction_type = "sample"

# ==== Prepare and load checkpoint ====
adapter, unet = accelerator.prepare(adapter, unet)
accelerator.load_state(ckpt_path)
adapter.eval(); unet.eval()

# ==== Load latent feature ====
z = torch.load(latent_path, map_location="cpu").to(device=device, dtype=torch.float16)
if z.ndim == 3:
    z = z.unsqueeze(0)  # ensure shape [1,4,64,64]

# ==== Decode with diffusion ====
with torch.no_grad():
    cond_feats = adapter(z)
    B, _, H, W = 1, 3, 512, 512
    x = torch.randn(B, 3, H, W, device=device) * scheduler.init_noise_sigma

    for t in scheduler.timesteps:
        t_batch = torch.full((B,), int(t), device=device, dtype=torch.long)
        x0_pred = unet(x, t_batch, cond_feats)
        x = scheduler.step(x0_pred, t, x).prev_sample

    out_img = (x.clamp(-1, 1) + 1) / 2  # to [0,1]

# ==== Save as PNG ====
to_pil = transforms.ToPILImage()
out_pil = to_pil(out_img[0].float().cpu())
out_pil.save(os.path.join(save_dir, "decoded_from_latent.png"))
print(f"Saved decoded PNG to: {save_dir}/decoded_from_latent.png")
